# Module 9 · 音訊 4：案例 — 環境聲音分類（現代管線）

> **本節定位｜整合案例**
> 把音訊 `01–03`（波形、log-mel、預訓練嵌入）整合到**聲音事件分類**。
> 示範 2026 常見作法：**預訓練音訊模型抽嵌入 + 傳統分類器**。
> 全程 CPU 可跑，使用**真實音訊**（小號音樂 vs 英語語音）。

## 學習目標
- 建立音訊分類流程：載入 → 標準化(16k/mono) → 抽嵌入 → 分類 → 評估。
- 對比「經典 MFCC 特徵」與「預訓練音訊嵌入」。
- 理解「想再更好 → 端到端微調」是下一步（M11）。

<!-- concept-image:04_urban_sound_case_fig1 -->
<div align="center">
<img src="concept_images/04_urban_sound_case_fig1_pipeline_20260611_222053.png" alt="聲音分類全流程" width="640" style="max-width:100%;">
<br><sub>圖 1 · 聲音分類全流程</sub>
</div>

<!-- concept-image:04_urban_sound_case_fig2 -->
<div align="center">
<img src="concept_images/04_urban_sound_case_fig2_mfcc_route_20260611_222129.png" alt="路線 A：MFCC 手工特徵" width="640" style="max-width:100%;">
<br><sub>圖 2 · 路線 A：MFCC 手工特徵</sub>
</div>

In [ ]:
import numpy as np
import librosa

# 真實音訊兩類：小號音樂 vs 英語語音（librosa 內建錄音，各切成 1 秒真實片段）
# 真實世界的對應任務如 UrbanSound8K / ESC-50；此處用內建真實音訊示範完整流程。
def load_audio_samples(n_per_class=5, sr=16000):
    music, _ = librosa.load(librosa.example("trumpet"), sr=sr)   # 真實小號獨奏
    speech, _ = librosa.load(librosa.example("libri1"), sr=sr)   # 真實英語語音
    def slice_clips(y, n, length):
        clips = []
        for i in range(n):
            seg = y[i*length:(i+1)*length]
            if len(seg) < length:
                seg = np.pad(seg, (0, length - len(seg)))
            clips.append(seg.astype(np.float32))
        return clips
    c0 = slice_clips(music, n_per_class, sr)    # 類別 0：音樂
    c1 = slice_clips(speech, n_per_class, sr)   # 類別 1：語音
    waves = c0 + c1
    labels = np.array([0]*n_per_class + [1]*n_per_class)
    return waves, labels, sr, "真實音訊：小號音樂 vs 英語語音（各切 1 秒片段）"

waves, labels, sr, source = load_audio_samples()
print(f"資料來源：{source}")
print(f"樣本數 = {len(waves)}（音樂 {int((labels==0).sum())} + 語音 {int((labels==1).sum())}，每段 1 秒、16kHz）")

## 路線 A（經典）：MFCC 聚合特徵 + 邏輯回歸

In [ ]:
try:
    import librosa
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    def mfcc_feat(w):
        m = librosa.feature.mfcc(y=w, sr=sr, n_mfcc=13)
        return np.concatenate([m.mean(axis=1), m.std(axis=1)])   # (26,)
    X_mfcc = np.stack([mfcc_feat(w) for w in waves])
    print(f"MFCC 特徵矩陣: {X_mfcc.shape}")
    acc = cross_val_score(LogisticRegression(max_iter=1000), X_mfcc, labels, cv=3).mean()
    print(f"[MFCC + LogReg] 交叉驗證 Accuracy = {acc:.3f}")
except Exception as e:
    print("（未安裝 librosa，略過路線 A）", e)

## 路線 B（現代）：預訓練音訊嵌入 + 邏輯回歸

<!-- concept-image:04_urban_sound_case_fig3 -->
<div align="center">
<img src="concept_images/04_urban_sound_case_fig3_comparison_20260611_183418.png" alt="路線 B：預訓練嵌入 vs 路線 A 性能對比" width="640" style="max-width:100%;">
<br><sub>圖 3 · 路線 B：預訓練嵌入 vs 路線 A 性能對比</sub>
</div>

In [ ]:
try:
    import torch
    from transformers import AutoFeatureExtractor, AutoModel
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    fe = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
    model = AutoModel.from_pretrained("facebook/wav2vec2-base-960h")
    model.eval()

    embs = []
    with torch.no_grad():
        for w in waves:
            inp = fe(w, sampling_rate=sr, return_tensors="pt")
            emb = model(**inp).last_hidden_state.mean(dim=1).squeeze(0)   # (D,)
            embs.append(emb.numpy())
    X_emb = np.stack(embs)
    print(f"音訊嵌入矩陣: {X_emb.shape}  (N, D)")
    acc = cross_val_score(LogisticRegression(max_iter=1000), X_emb, labels, cv=3).mean()
    print(f"[wav2vec2 嵌入 + LogReg] 交叉驗證 Accuracy = {acc:.3f}")
except Exception as e:
    print("（未能下載 wav2vec2，略過路線 B）", e)

## 小結與延伸

| 路線 | 特徵 | 特性 |
|:--|:--|:--|
| A. MFCC + LogReg | 手工聚合特徵 | 輕量、可解釋，無語意 |
| B. wav2vec2 嵌入 + LogReg | 預訓練表示 | 帶語意、較穩健、可遷移 |

**想要最高準確率？** 兩條都是「凍結特徵」。把音訊模型（如 wav2vec2）
**端到端微調**做分類、或微調 **Whisper** 做語音辨識，是 **Module 11 · `04_audio_downstream`** 的內容。